# Day 5 — Unified FastAPI Backend & ngrok Deployment

**Goal**: Serve **all four models** (product classifier, sentiment, chatbot, face ID)
behind a single FastAPI app, then expose it publicly via **pyngrok**.

**Deliverable**: `05_day5_fastapi_deployment.ipynb`


In [ ]:
import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/Major_Project'
sys.path.append(PROJECT_ROOT); os.chdir(PROJECT_ROOT)

## 1. Sanity-Check Trained Artefacts

Notebooks 2-4 must have been executed at least once so that these files exist:

In [ ]:
for f in [
    'models/product_classifier_model.h5',
    'models/product_labels.json',
    'models/sentiment_model.joblib',
    'models/chatbot.joblib',
    'models/face_db.pkl',
]:
    print(('✅' if os.path.exists(f) else '❌'), f)

## 2. Start the FastAPI Server (background thread)

In [ ]:
import nest_asyncio, threading, uvicorn
from src.api import app

nest_asyncio.apply()

def _run():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

threading.Thread(target=_run, daemon=True).start()
print('FastAPI running on http://localhost:8000')

## 3. Expose Publicly via ngrok

Grab a free auth token at <https://dashboard.ngrok.com/get-started/your-authtoken>.

In [ ]:
NGROK_AUTHTOKEN = 'paste_your_token_here'

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_AUTHTOKEN
public_url = ngrok.connect(8000).public_url
print('🔗 Public URL:', public_url)
print('   Swagger UI :', public_url + '/docs')

## 4. End-to-End API Tests

In [ ]:
import requests, time

BASE = 'http://localhost:8000'
time.sleep(2)

print('HEALTH   ->', requests.get(f'{BASE}/health').json())
print('CHAT     ->', requests.post(f'{BASE}/chatbot',
        json={'query': 'How do I return an item?'}).json())
print('SENTIMENT->', requests.post(f'{BASE}/analyze-sentiment',
        json={'text': 'Absolutely love this product!'}).json())

with open('data/products/shoes/img_00.jpg', 'rb') as f:
    r = requests.post(f'{BASE}/classify-product', files={'file': f})
print('PRODUCT  ->', r.json())

with open('data/faces/alice/img_1.jpg', 'rb') as f:
    r = requests.post(f'{BASE}/identify-customer', files={'file': f})
print('FACE     ->', r.json())

## 5. Shutdown (when done)

```python
ngrok.disconnect(public_url)
ngrok.kill()
```

🎉 **Project complete!** Your unified AI Smart Retail platform is now live and testable
from any device via the ngrok public URL.